# 02 · Ask the data questions in SQL
**Session 3 · step 2 of 4 · ~20 min**

🧠 **The idea.** Governed gold means the hard modeling is done — so an analyst's SQL is
short and reads like the business question. We'll land the four story beats: **concentration,
collateral, credit (MPF), and housing.** Try each yourself; a reference answer follows.

🏢 **Why FHLB-Topeka cares.** These are the Bank's standing questions. Answering them from
governed gold in a few lines is exactly the "analyst enablement" outcome for the day.

In [ ]:
# ============================================================
#  WORKSHOP CONFIG (1 of 2)  —  pick your catalog, THEN run the next cell
# ============================================================
# Mode A (live, our workspace):  serverless_stable_6fhczt_catalog  (the default)
# Mode B (portable / Free Edition): type the catalog 00_LOAD_DATA used
#   (the one you created, or an existing one you loaded into). Same value in every module.
# This cell only creates the picker at the top of the notebook.
dbutils.widgets.text("catalog", "serverless_stable_6fhczt_catalog", "Catalog")
print("↑ Set the 'Catalog' widget at the top of the notebook, then run the next cell.")

In [ ]:
# ---- WORKSHOP CONFIG (2 of 2)  —  apply the selected catalog ----
CATALOG = dbutils.widgets.get("catalog").strip()
assert CATALOG, "Set the 'Catalog' widget at the top of the notebook, then re-run this cell."

GOLD   = f"{CATALOG}.fhlb_gold"     # governed, analyst-ready data products (read-only)
SILVER = f"{CATALOG}.fhlb_silver"   # cleaned/typed layer (we use the HPI time series here)

spark.sql(f"USE CATALOG {CATALOG}")
print(f"Catalog: {CATALOG}  ·  gold: {GOLD}")

## Beat 1 — Concentration: who dominates the advance book?
**Predict-then-check:** what share do you think the single largest member holds? 5%? 15%? 25%?

In [ ]:
%sql
-- Try it: rank members by outstanding advances and show each one's share of the book.
-- (portfolio_concentration already has share_of_advance_book.)
-- Write your query, then compare to the reference below.

In [ ]:
%sql
-- ▼ Reference answer — compare after you try
SELECT member_name, state,
       ROUND(total_outstanding_par/1e6, 1)  AS outstanding_$m,
       ROUND(share_of_advance_book*100, 1)  AS pct_of_book
FROM fhlb_gold.portfolio_concentration
ORDER BY total_outstanding_par DESC
LIMIT 5

👀 **Insight.** **Midwest Savings Bank (NE) ≈ 24.3% ($335M)** of a **$1.38B** book;
the **top 5 ≈ 62%.** One member is nearly a quarter of the book — that's concentration risk
you'd want a limit and a dashboard tile on.

## Beat 2 — Collateral: who's undercollateralized?
Averages say ~30% utilization (healthy). Averages lie. Find the exceptions.

In [ ]:
%sql
-- Try it: which members have collateral utilization over 100% (advances > lendable value)?
-- table: fhlb_gold.member_collateral_capacity, column: collateral_utilization_pct (a ratio, 1.0 = 100%)

In [ ]:
%sql
-- ▼ Reference answer
SELECT member_id,
       ROUND(collateral_utilization_pct*100, 1) AS utilization_pct,
       ROUND(total_outstanding_par/1e6, 1)      AS advances_$m,
       ROUND(total_lendable_value/1e6, 1)       AS lendable_$m,
       is_undercollateralized,
       CASE WHEN stale_market_value > 0 THEN 'STALE' ELSE '' END AS valuation_flag
FROM fhlb_gold.member_collateral_capacity
ORDER BY collateral_utilization_pct DESC
LIMIT 8

👀 **Insight.** **M1011 is undercollateralized at ~106.6%** — advances exceed lendable
value. And **7 members carry a stale market value** on collateral: a data-quality signal that
is itself a risk (you may be lending against a valuation that's out of date).

## Beat 3 — Credit: is MPF delinquency uniform, or by product?

In [ ]:
%sql
-- ▼ Reference answer — delinquency by MPF product
SELECT mpf_product,
       SUM(loan_count)                       AS loans,
       ROUND(AVG(delinquency_rate)*100, 2)   AS delinquency_pct,
       ROUND(AVG(serious_delinquency_rate)*100, 2) AS serious_pct,
       ROUND(AVG(avg_credit_score))          AS avg_fico
FROM fhlb_gold.mpf_portfolio_summary
GROUP BY mpf_product
ORDER BY delinquency_pct DESC

👀 **Insight.** Overall delinquency is **4.76%**, but it's **not uniform**:
**MPF Xtra ≈ 5.71%** vs **MPF 35 ≈ 3.21%.** Product mix, not just the headline, drives credit risk.

## Beat 4 — Housing: the district's HPI trend (from the silver time series)
The gold `housing_market_reference` is a *point-in-time* snapshot. For a trend, use the
silver HPI time series.

In [ ]:
%sql
-- ▼ Reference answer — district (CO/KS/NE/OK) average HPI by year
SELECT year, ROUND(AVG(hpi_index), 1) AS avg_hpi
FROM fhlb_silver.fhfa_hpi
WHERE state IN ('CO','KS','NE','OK')
GROUP BY year
ORDER BY year

👀 **Insight.** District HPI ran **58.8 (1975) → 569.4 (2026)**, with a visible **dip
through 2008–2011** and a steep **2020–2023** climb. Note we trend on complete years — the newest
partial period can read flat/0% YoY and should be labeled, not charted as a cliff.

## 🧑‍💻 Your Turn (pick your path)
- **SQL:** join `member_advance_summary` to `member_collateral_capacity` — is the most
  *concentrated* member also well-collateralized?
- **Or skip ahead:** you'll ask exactly these in plain English in module 03.

## ⚠️ Fallback
Every reference cell above is runnable as-is against governed gold — run them if your own
query misbehaves.

## 🌟 Optional
Rank members by `par_maturing_90d` (from `member_advance_summary`) — who has the most
advances rolling off in the next quarter?

---
### ✅ Done
**Next:** open `03_genie_space` — now you'll ask these in plain English.